In [14]:
#importamos librerías necesarias
import pandas as pd
import requests
import os
from dotenv import load_dotenv
import tarfile
import gzip
import io
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import time
import re

In [15]:
# cargamos API_KEY
load_dotenv()

API_KEY = os.getenv("AEMET_API_KEY")

if API_KEY is None:
    raise ValueError("No se ha encontrado la API Key. Revisa el archivo .env")
else:
    print("API Key cargada correctamente")

python-dotenv could not parse statement starting at line 2


API Key cargada correctamente


In [17]:
# definimos fechas del proyecto 
fecha_inicio_total = datetime(2021, 1, 1)
fecha_fin_total = datetime(2026, 4, 30)

print("Fecha inicio:", fecha_inicio_total)
print("Fecha fin:", fecha_fin_total)

Fecha inicio: 2021-01-01 00:00:00
Fecha fin: 2026-04-30 00:00:00


In [18]:
# Creamos rango de dos dias para la extracción de datos
rangos_fechas = []

fecha_actual = fecha_inicio_total

while fecha_actual <= fecha_fin_total:
    inicio = fecha_actual
    fin = min(fecha_actual + timedelta(days=1), fecha_fin_total)

    rangos_fechas.append((inicio, fin))

    fecha_actual = fin + timedelta(days=1)

print("Número de rangos creados:", len(rangos_fechas))

for inicio, fin in rangos_fechas[:10]:
    print(inicio.strftime("%Y-%m-%d"), "→", fin.strftime("%Y-%m-%d"))

print("...")

for inicio, fin in rangos_fechas[-10:]:
    print(inicio.strftime("%Y-%m-%d"), "→", fin.strftime("%Y-%m-%d"))

Número de rangos creados: 973
2021-01-01 → 2021-01-02
2021-01-03 → 2021-01-04
2021-01-05 → 2021-01-06
2021-01-07 → 2021-01-08
2021-01-09 → 2021-01-10
2021-01-11 → 2021-01-12
2021-01-13 → 2021-01-14
2021-01-15 → 2021-01-16
2021-01-17 → 2021-01-18
2021-01-19 → 2021-01-20
...
2026-04-11 → 2026-04-12
2026-04-13 → 2026-04-14
2026-04-15 → 2026-04-16
2026-04-17 → 2026-04-18
2026-04-19 → 2026-04-20
2026-04-21 → 2026-04-22
2026-04-23 → 2026-04-24
2026-04-25 → 2026-04-26
2026-04-27 → 2026-04-28
2026-04-29 → 2026-04-30


In [19]:
# funciones auxiliares para leer el XML de AEMET
def limpiar_tag(tag):
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def hijos(elemento, nombre):
    return [hijo for hijo in list(elemento) if limpiar_tag(hijo.tag) == nombre]


def texto(elemento, nombre):
    for hijo in list(elemento):
        if limpiar_tag(hijo.tag) == nombre:
            return hijo.text if hijo.text is not None else ""
    return ""

In [20]:
#funciones para descargar un rango de 2 dias 
def descargar_rango_aemet(fecha_inicio, fecha_fin, api_key, pausa=10, max_intentos=5):
    fecha_ini = fecha_inicio.strftime("%Y-%m-%dT00:00:00UTC")
    fecha_fin_str = fecha_fin.strftime("%Y-%m-%dT23:59:59UTC")

    url = (
        "https://opendata.aemet.es/opendata/api/avisos_cap/archivo/"
        f"fechaini/{fecha_ini}/fechafin/{fecha_fin_str}"
    )

    params = {
        "api_key": API_KEY
    }

    for intento in range(1, max_intentos + 1):
        print(f"Intento {intento}/{max_intentos}: {fecha_ini} → {fecha_fin_str}")

        # Primera llamada: pedir a AEMET la URL temporal
        response = requests.get(url, params=params, timeout=90)

        if response.status_code == 429:
            print("Demasiadas peticiones. Esperando 90 segundos...")
            time.sleep(90)
            continue

        if response.status_code != 200:
            print("Error en primera petición:", response.status_code)
            print(response.text[:500])
            time.sleep(pausa)
            continue

        try:
            respuesta_json = response.json()
        except Exception:
            print("La respuesta no es JSON")
            print(response.text[:500])
            time.sleep(pausa)
            continue

        url_datos = respuesta_json.get("datos")

        if not url_datos:
            print("No se encontró URL de datos")
            print(respuesta_json)
            time.sleep(pausa)
            continue

        time.sleep(pausa)

        # Segunda llamada: descargar el archivo real
        datos_response = requests.get(url_datos)

        if datos_response.status_code == 429:
            print("Demasiadas peticiones en descarga. Esperando 90 segundos...")
            time.sleep(90)
            continue

        if datos_response.status_code == 500:
            print("Error 500 en AEMET. Esperando y reintentando...")
            time.sleep(60)
            continue

        if datos_response.status_code != 200:
            print("Error descargando archivo real:", datos_response.status_code)
            print(datos_response.text[:500])
            time.sleep(pausa)
            continue

        contenido = datos_response.content

        # Comprobación mínima: parece TAR si empieza con nombre de archivo Z_CAP...
        if not contenido.startswith(b"Z_CAP"):
            print("El contenido descargado no parece el archivo esperado.")
            print(contenido[:300])
            time.sleep(pausa)
            continue

        print("Descarga correcta")
        return contenido

    print("No se pudo descargar este rango después de varios intentos.")
    return None

In [21]:
# Función para convertir TAR + GZ + XML en tabla
def convertir_contenido_a_dataframe(contenido):
    filas = []

    try:
        archivo_tar = tarfile.open(fileobj=io.BytesIO(contenido), mode="r:")
    except tarfile.TarError:
        print("No se pudo abrir como TAR")
        return pd.DataFrame()

    for miembro in archivo_tar.getmembers():

        if not miembro.name.endswith(".gz"):
            continue

        archivo_extraido = archivo_tar.extractfile(miembro)

        if archivo_extraido is None:
            continue

        contenido_gz = archivo_extraido.read()

        try:
            contenido_xml = gzip.decompress(contenido_gz)
        except gzip.BadGzipFile:
            continue

        texto_xml = contenido_xml.decode("utf-8", errors="ignore")

        bloques_xml = re.findall(
            r"<\?xml.*?</alert>",
            texto_xml,
            flags=re.DOTALL
        )

        for bloque in bloques_xml:

            try:
                root = ET.fromstring(bloque)
            except ET.ParseError:
                continue

            if limpiar_tag(root.tag) == "alert":
                alertas = [root]
            else:
                alertas = [
                    elem for elem in root.iter()
                    if limpiar_tag(elem.tag) == "alert"
                ]

            for alert in alertas:

                identifier = texto(alert, "identifier")
                sender = texto(alert, "sender")
                sent = texto(alert, "sent")
                status = texto(alert, "status")
                msg_type = texto(alert, "msgType")
                scope = texto(alert, "scope")

                for info in hijos(alert, "info"):

                    language = texto(info, "language")
                    category = texto(info, "category")
                    event = texto(info, "event")
                    urgency = texto(info, "urgency")
                    severity = texto(info, "severity")
                    certainty = texto(info, "certainty")
                    effective = texto(info, "effective")
                    onset = texto(info, "onset")
                    expires = texto(info, "expires")
                    headline = texto(info, "headline")
                    description = texto(info, "description")

                    areas = hijos(info, "area")

                    for area in areas:

                        area_desc = texto(area, "areaDesc")
                        polygon = texto(area, "polygon")
                        circle = texto(area, "circle")

                        geocode_valores = []

                        for geocode in hijos(area, "geocode"):
                            value_name = texto(geocode, "valueName")
                            value = texto(geocode, "value")

                            if value_name or value:
                                geocode_valores.append(f"{value_name}: {value}")

                        geocode_texto = " | ".join(geocode_valores)

                        filas.append({
                            "identifier": identifier,
                            "sender": sender,
                            "sent": sent,
                            "status": status,
                            "msg_type": msg_type,
                            "scope": scope,
                            "language": language,
                            "category": category,
                            "event": event,
                            "urgency": urgency,
                            "severity": severity,
                            "certainty": certainty,
                            "effective": effective,
                            "onset": onset,
                            "expires": expires,
                            "headline": headline,
                            "description": description,
                            "area_desc": area_desc,
                            "polygon": polygon,
                            "circle": circle,
                            "geocode": geocode_texto,
                            "archivo_origen": miembro.name
                        })

    return pd.DataFrame(filas)

In [22]:
# funciones para limpiar y crear nuevas columnas 
def limpiar_dataframe(df):
    if df.empty:
        return df

    df_limpio = df.drop_duplicates(
        subset=[
            "identifier",
            "sent",
            "onset",
            "expires",
            "event",
            "severity",
            "area_desc",
            "headline"
        ]
    ).copy()

    # Convertimos fechas usando utc=True para evitar errores con zonas horarias mixtas
    df_limpio["onset"] = pd.to_datetime(df_limpio["onset"], errors="coerce", utc=True)
    df_limpio["expires"] = pd.to_datetime(df_limpio["expires"], errors="coerce", utc=True)
    df_limpio["sent"] = pd.to_datetime(df_limpio["sent"], errors="coerce", utc=True)

    # Creamos año y mes
    df_limpio["año"] = df_limpio["onset"].dt.year
    df_limpio["mes_num"] = df_limpio["onset"].dt.month

    nombres_meses = {
        1: "Enero",
        2: "Febrero",
        3: "Marzo",
        4: "Abril",
        5: "Mayo",
        6: "Junio",
        7: "Julio",
        8: "Agosto",
        9: "Septiembre",
        10: "Octubre",
        11: "Noviembre",
        12: "Diciembre"
    }

    df_limpio["mes"] = df_limpio["mes_num"].map(nombres_meses)

    df_limpio["estacion"] = df_limpio["mes_num"].map({
        12: "Invierno",
        1: "Invierno",
        2: "Invierno",
        3: "Primavera",
        4: "Primavera",
        5: "Primavera",
        6: "Verano",
        7: "Verano",
        8: "Verano",
        9: "Otoño",
        10: "Otoño",
        11: "Otoño"
    })

    df_limpio["duracion_horas"] = (
        df_limpio["expires"] - df_limpio["onset"]
    ).dt.total_seconds() / 3600

    df_limpio["riesgo_numerico"] = df_limpio["severity"].map({
        "Minor": 0,
        "Moderate": 1,
        "Severe": 2,
        "Extreme": 3
    }).fillna(0)

    return df_limpio

In [9]:
# probamos con un solo rango antes que todo 
contenido_prueba = descargar_rango_aemet(
    datetime(2021, 1, 7),
    datetime(2021, 1, 8),
    API_KEY,
    pausa=12
)

df_prueba = convertir_contenido_a_dataframe(contenido_prueba)

df_prueba_limpio = limpiar_dataframe(df_prueba)

print("Filas prueba:", len(df_prueba_limpio))

df_prueba_limpio.head()

Intento 1/5: 2021-01-07T00:00:00UTC → 2021-01-08T23:59:59UTC
Descarga correcta
Filas prueba: 12604


,identifier,sender,sent,status,msg_type,scope,language,category,event,urgency,...,polygon,circle,geocode,archivo_origen,año,mes_num,mes,estacion,duracion_horas,riesgo_numerico
0,2.49.0.0.724.0.ES.20210107043640.750102BTTI070...,http://www.aemet.es,2021-01-07 04:36:40+00:00,Actual,Alert,Public,es-ES,Met,Aviso de temperaturas mínimas de nivel amarillo,Immediate,...,"42.94,-2.98 42.94,-2.93 42.91,-2.86 42.92,-2.7...",,AEMET-Meteoalerta zona: 750102,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,3.999722,1
1,2.49.0.0.724.0.ES.20210107043640.750102BTTI070...,http://www.aemet.es,2021-01-07 04:36:40+00:00,Actual,Alert,Public,en-GB,Met,Moderate low-temperature warning,Immediate,...,"42.94,-2.98 42.94,-2.93 42.91,-2.86 42.92,-2.7...",,AEMET-Meteoalerta zona: 750102,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,3.999722,1
2,2.49.0.0.724.0.ES.20210106225001.624401NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,es-ES,Met,Aviso de nevadas de nivel naranja,Future,...,"40.94,-1.62 40.94,-1.56 40.96,-1.5 40.96,-1.47...",,AEMET-Meteoalerta zona: 624401,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2
3,2.49.0.0.724.0.ES.20210106225001.624401NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,en-GB,Met,Severe snow warning,Future,...,"40.94,-1.62 40.94,-1.56 40.96,-1.5 40.96,-1.47...",,AEMET-Meteoalerta zona: 624401,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2
4,2.49.0.0.724.0.ES.20210106225001.624402NENV092...,http://www.aemet.es,2021-01-06 22:50:01+00:00,Actual,Alert,Public,es-ES,Met,Aviso de nevadas de nivel naranja,Future,...,"39.97,-1.14 40.01,-1.16 40.04,-1.08 40.06,-1.0...",,AEMET-Meteoalerta zona: 624402,Z_CAP_C_LEMM_20210107043640_AFAE.gz,2021,1,Enero,Invierno,23.999722,2


In [23]:
# Crear carpeta por meses 
import os

carpeta_salida = "avisos_aemet_por_mes"

os.makedirs(carpeta_salida, exist_ok=True)

print("Carpeta lista:", carpeta_salida)

Carpeta lista: avisos_aemet_por_mes


In [20]:
# creamos la lista de meses desde enero de 2021 hasta abril de 2026
meses_a_procesar = []

for anio in range(2021, 2027):
    for mes in range(1, 13):

        if anio == 2026 and mes > 4:
            continue

        meses_a_procesar.append((anio, mes))

print("Número de meses:", len(meses_a_procesar))
print(meses_a_procesar[:5])
print(meses_a_procesar[-5:])

Número de meses: 64
[(2021, 1), (2021, 2), (2021, 3), (2021, 4), (2021, 5)]
[(2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4)]


In [24]:
meses_a_procesar = []

for anio in range(2023, 2027):
    for mes in range(1, 13):

        if anio == 2023 and mes < 6:
            continue

        if anio == 2026 and mes > 4:
            continue

        meses_a_procesar.append((anio, mes))

print("Meses pendientes:", len(meses_a_procesar))
print(meses_a_procesar[:5])
print(meses_a_procesar[-5:])

Meses pendientes: 35
[(2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10)]
[(2025, 12), (2026, 1), (2026, 2), (2026, 3), (2026, 4)]


In [25]:
# descargar y guardar un CSV por cada mes
archivos_generados = []

for anio, mes in meses_a_procesar:

    print("=" * 80)
    print(f"PROCESANDO {anio}-{mes:02d}")
    print("=" * 80)

    # Primer y último día del mes
    fecha_inicio_mes = datetime(anio, mes, 1)

    if mes == 12:
        fecha_inicio_siguiente_mes = datetime(anio + 1, 1, 1)
    else:
        fecha_inicio_siguiente_mes = datetime(anio, mes + 1, 1)

    fecha_fin_mes = fecha_inicio_siguiente_mes - timedelta(days=1)

    # Dividir el mes en bloques de máximo 2 días
    rangos_mes = []
    fecha_actual = fecha_inicio_mes

    while fecha_actual <= fecha_fin_mes:
        inicio = fecha_actual
        fin = min(fecha_actual + timedelta(days=1), fecha_fin_mes)

        rangos_mes.append((inicio, fin))

        fecha_actual = fin + timedelta(days=1)

    print("Rangos de 2 días en este mes:", len(rangos_mes))

    dfs_mes = []

    for inicio, fin in rangos_mes:

        contenido = descargar_rango_aemet(
            inicio,
            fin,
            API_KEY,
            pausa=8,
            max_intentos=5
        )

        if contenido is None:
            print("No se pudo descargar este rango. Se pasa al siguiente.")
            continue

        df_rango = convertir_contenido_a_dataframe(contenido)

        if df_rango.empty:
            print("No se encontraron datos en este rango.")
            continue

        df_rango_limpio = limpiar_dataframe(df_rango)

        if df_rango_limpio.empty:
            print("El rango quedó vacío después de limpiar.")
            continue

        dfs_mes.append(df_rango_limpio)

        print("Filas limpias del rango:", len(df_rango_limpio))

    if len(dfs_mes) == 0:
        print(f"No se obtuvieron datos para {anio}-{mes:02d}")
        continue

    # Unir todos los rangos del mes
    df_mes = pd.concat(dfs_mes, ignore_index=True)

    # Quitar duplicados dentro del mes
    df_mes_limpio = df_mes.drop_duplicates(
        subset=[
            "identifier",
            "sent",
            "onset",
            "expires",
            "event",
            "severity",
            "area_desc",
            "headline"
        ]
    ).copy()

    print("Filas del mes antes de limpieza final:", len(df_mes))
    print("Filas del mes después de limpieza final:", len(df_mes_limpio))

    nombre_archivo = f"{carpeta_salida}/avisos_aemet_{anio}_{mes:02d}.csv"

    df_mes_limpio.to_csv(
        nombre_archivo,
        index=False,
        encoding="utf-8-sig"
    )

    archivos_generados.append(nombre_archivo)

    print("Archivo mensual guardado:", nombre_archivo)

PROCESANDO 2023-06
Rangos de 2 días en este mes: 15
Intento 1/5: 2023-06-01T00:00:00UTC → 2023-06-02T23:59:59UTC
Descarga correcta
Filas limpias del rango: 11788
Intento 1/5: 2023-06-03T00:00:00UTC → 2023-06-04T23:59:59UTC
Descarga correcta
Filas limpias del rango: 11264
Intento 1/5: 2023-06-05T00:00:00UTC → 2023-06-06T23:59:59UTC
Descarga correcta
Filas limpias del rango: 10388
Intento 1/5: 2023-06-07T00:00:00UTC → 2023-06-08T23:59:59UTC
Descarga correcta
Filas limpias del rango: 10076
Intento 1/5: 2023-06-09T00:00:00UTC → 2023-06-10T23:59:59UTC
Descarga correcta
Filas limpias del rango: 10052
Intento 1/5: 2023-06-11T00:00:00UTC → 2023-06-12T23:59:59UTC
Demasiadas peticiones. Esperando 90 segundos...
Intento 2/5: 2023-06-11T00:00:00UTC → 2023-06-12T23:59:59UTC
Descarga correcta
Filas limpias del rango: 10958
Intento 1/5: 2023-06-13T00:00:00UTC → 2023-06-14T23:59:59UTC
Descarga correcta
Filas limpias del rango: 10188
Intento 1/5: 2023-06-15T00:00:00UTC → 2023-06-16T23:59:59UTC
Descarga

In [26]:
# comprobar archivos mensuales generados
print("Número de archivos mensuales generados:", len(archivos_generados))

archivos_generados[:10]

Número de archivos mensuales generados: 35


['avisos_aemet_por_mes/avisos_aemet_2023_06.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_07.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_08.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_09.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_10.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_11.csv',
 'avisos_aemet_por_mes/avisos_aemet_2023_12.csv',
 'avisos_aemet_por_mes/avisos_aemet_2024_01.csv',
 'avisos_aemet_por_mes/avisos_aemet_2024_02.csv',
 'avisos_aemet_por_mes/avisos_aemet_2024_03.csv']

In [32]:

import glob

archivos_generados = sorted(glob.glob("avisos_aemet_por_mes/*.csv"))

print("Archivos encontrados:", len(archivos_generados))


Archivos encontrados: 64


In [38]:
# definimos las columnas 
columnas_utiles = [
    "identifier",
    "sent",
    "onset",
    "expires",
    "año",
    "mes",
    "mes_num",
    "estacion",
    "event",
    "severity",
    "riesgo_numerico",
    "duracion_horas",
    "area_desc",
    "polygon",
    "geocode",
    "headline",
    "description"
]

columnas_duplicados = [
    "identifier",
    "sent",
    "onset",
    "expires",
    "event",
    "severity",
    "area_desc",
    "headline"
]

In [39]:
archivo_final = "avisos_aemet_2021_2026_reducido_tableau.csv"

# Si ya existe, lo borramos para empezar limpio
if os.path.exists(archivo_final):
    os.remove(archivo_final)

primera_vez = True
total_filas_guardadas = 0

for archivo in archivos_generados:
    print("=" * 80)
    print("Procesando:", archivo)
    print("=" * 80)

    try:
        lector = pd.read_csv(
            archivo,
            chunksize=50000,
            encoding="utf-8-sig",
            low_memory=False
        )

        for i, chunk in enumerate(lector):
            print(f"  Bloque {i + 1}")

            # Nos quedamos solo con las columnas útiles que existan
            columnas_disponibles = [
                col for col in columnas_utiles
                if col in chunk.columns
            ]

            chunk = chunk[columnas_disponibles]

            # Quitamos duplicados dentro del bloque
            columnas_dup_disponibles = [
                col for col in columnas_duplicados
                if col in chunk.columns
            ]

            chunk = chunk.drop_duplicates(
                subset=columnas_dup_disponibles
            )

            # Guardamos el bloque en el archivo final
            chunk.to_csv(
                archivo_final,
                mode="w" if primera_vez else "a",
                header=primera_vez,
                index=False,
                encoding="utf-8-sig"
            )

            primera_vez = False
            total_filas_guardadas += len(chunk)

            print("  Filas añadidas:", len(chunk))
            print("  Total acumulado:", total_filas_guardadas)

    except Exception as e:
        print("ERROR leyendo archivo:")
        print(archivo)
        print(e)
        continue

print("Proceso terminado")
print("CSV final creado:", archivo_final)
print("Filas guardadas aproximadamente:", total_filas_guardadas)

Procesando: avisos_aemet_por_mes\avisos_aemet_2021_01.csv
  Bloque 1
  Filas añadidas: 50000
  Total acumulado: 50000
  Bloque 2
  Filas añadidas: 50000
  Total acumulado: 100000
  Bloque 3
  Filas añadidas: 17594
  Total acumulado: 117594
Procesando: avisos_aemet_por_mes\avisos_aemet_2021_02.csv
  Bloque 1
  Filas añadidas: 50000
  Total acumulado: 167594
  Bloque 2
  Filas añadidas: 46816
  Total acumulado: 214410
Procesando: avisos_aemet_por_mes\avisos_aemet_2021_03.csv
  Bloque 1
  Filas añadidas: 50000
  Total acumulado: 264410
  Bloque 2
  Filas añadidas: 50000
  Total acumulado: 314410
  Bloque 3
  Filas añadidas: 2336
  Total acumulado: 316746
Procesando: avisos_aemet_por_mes\avisos_aemet_2021_04.csv
  Bloque 1
  Filas añadidas: 50000
  Total acumulado: 366746
  Bloque 2
  Filas añadidas: 49180
  Total acumulado: 415926
Procesando: avisos_aemet_por_mes\avisos_aemet_2021_05.csv
  Bloque 1
  Filas añadidas: 50000
  Total acumulado: 465926
  Bloque 2
  Filas añadidas: 50000
  Tota

In [40]:
df_muestra = pd.read_csv(
    "avisos_aemet_2021_2026_reducido_tableau.csv",
    nrows=1000,
    encoding="utf-8-sig"
)

print(df_muestra.shape)
df_muestra.head()

(1000, 17)


,identifier,sent,onset,expires,año,mes,mes_num,estacion,event,severity,riesgo_numerico,duracion_horas,area_desc,polygon,geocode,headline,description
0,2.49.0.0.724.0.ES.20210101064907.610401VIRM012...,2021-01-01 06:49:07+00:00,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,13.999722,Valle del Almanzora y Los Vélez,"37.92,-2.21 37.89,-2.17 37.9,-2.12 37.88,-2.1 ...",AEMET-Meteoalerta zona: 610401,Aviso de vientos de nivel amarillo. Valle del ...,Rachas máximas: 70 km/h. Vientos del oeste
1,2.49.0.0.724.0.ES.20210101064907.610401VIRM012...,2021-01-01 06:49:07+00:00,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Moderate wind warning,Moderate,1,13.999722,Valle del Almanzora y Los Vélez,"37.92,-2.21 37.89,-2.17 37.9,-2.12 37.88,-2.1 ...",AEMET-Meteoalerta zona: 610401,Moderate wind warning. Valle del Almanzora y L...,Maximum gust of wind: 70 km/h. Vientos del oeste
2,2.49.0.0.724.0.ES.20210101064907.610402VIRM012...,2021-01-01 06:49:07+00:00,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,13.999722,Nacimiento y Campo de Tabernas,"37.1,-2.95 37.16,-2.93 37.17,-2.89 37.23,-2.87...",AEMET-Meteoalerta zona: 610402,Aviso de vientos de nivel amarillo. Nacimiento...,Rachas máximas: 70 km/h. Vientos del oeste
3,2.49.0.0.724.0.ES.20210101064907.610402VIRM012...,2021-01-01 06:49:07+00:00,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,2021,Enero,1,Invierno,Moderate wind warning,Moderate,1,13.999722,Nacimiento y Campo de Tabernas,"37.1,-2.95 37.16,-2.93 37.17,-2.89 37.23,-2.87...",AEMET-Meteoalerta zona: 610402,Moderate wind warning. Nacimiento y Campo de T...,Maximum gust of wind: 70 km/h. Vientos del oeste
4,2.49.0.0.724.0.ES.20210101064907.610403VIRM012...,2021-01-01 06:49:07+00:00,2021-01-01 06:00:00+00:00,2021-01-01 20:59:59+00:00,2021,Enero,1,Invierno,Aviso de vientos de nivel amarillo,Moderate,1,14.999722,Poniente y Almería Capital,"36.75,-3.13 36.79,-3.14 36.84,-3.03 36.92,-3.0...",AEMET-Meteoalerta zona: 610403,Aviso de vientos de nivel amarillo. Poniente y...,Rachas máximas: 70 km/h. Vientos del oeste


In [41]:
df_muestra.columns.tolist()

['identifier',
 'sent',
 'onset',
 'expires',
 'año',
 'mes',
 'mes_num',
 'estacion',
 'event',
 'severity',
 'riesgo_numerico',
 'duracion_horas',
 'area_desc',
 'polygon',
 'geocode',
 'headline',
 'description']

In [42]:
df_muestra[["onset", "expires", "event", "severity", "area_desc"]].head(20)

,onset,expires,event,severity,area_desc
0,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,Aviso de vientos de nivel amarillo,Moderate,Valle del Almanzora y Los Vélez
1,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,Moderate wind warning,Moderate,Valle del Almanzora y Los Vélez
2,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,Aviso de vientos de nivel amarillo,Moderate,Nacimiento y Campo de Tabernas
3,2021-01-01 06:00:00+00:00,2021-01-01 19:59:59+00:00,Moderate wind warning,Moderate,Nacimiento y Campo de Tabernas
4,2021-01-01 06:00:00+00:00,2021-01-01 20:59:59+00:00,Aviso de vientos de nivel amarillo,Moderate,Poniente y Almería Capital
5,2021-01-01 06:00:00+00:00,2021-01-01 20:59:59+00:00,Moderate wind warning,Moderate,Poniente y Almería Capital
6,2021-01-01 06:00:00+00:00,2021-01-01 18:59:59+00:00,Aviso de vientos de nivel amarillo,Moderate,Costa granadina
7,2021-01-01 06:00:00+00:00,2021-01-01 18:59:59+00:00,Moderate wind warning,Moderate,Costa granadina
8,2021-01-02 23:00:00+00:00,2021-01-03 07:59:59+00:00,Aviso de temperaturas mínimas de nivel naranja,Severe,Pirineo oscense
9,2021-01-02 23:00:00+00:00,2021-01-03 07:59:59+00:00,Severe low-temperature warning,Severe,Pirineo oscense


In [43]:
archivo_ligero = "avisos_aemet_2021_2026_tableau_ligero.csv"

columnas_ligeras = [
    "identifier",
    "sent",
    "onset",
    "expires",
    "año",
    "mes",
    "mes_num",
    "estacion",
    "event",
    "severity",
    "riesgo_numerico",
    "duracion_horas",
    "area_desc",
    "geocode"
]

if os.path.exists(archivo_ligero):
    os.remove(archivo_ligero)

primera_vez = True
total_filas_ligero = 0

lector = pd.read_csv(
    "avisos_aemet_2021_2026_reducido_tableau.csv",
    chunksize=50000,
    encoding="utf-8-sig",
    low_memory=False
)

for i, chunk in enumerate(lector):
    print(f"Procesando bloque {i + 1}")

    columnas_disponibles = [
        col for col in columnas_ligeras
        if col in chunk.columns
    ]

    chunk = chunk[columnas_disponibles]

    chunk.to_csv(
        archivo_ligero,
        mode="w" if primera_vez else "a",
        header=primera_vez,
        index=False,
        encoding="utf-8-sig"
    )

    primera_vez = False
    total_filas_ligero += len(chunk)

print("CSV ligero creado:", archivo_ligero)
print("Filas:", total_filas_ligero)

Procesando bloque 1
Procesando bloque 2
Procesando bloque 3
Procesando bloque 4
Procesando bloque 5
Procesando bloque 6
Procesando bloque 7
Procesando bloque 8
Procesando bloque 9
Procesando bloque 10
Procesando bloque 11
Procesando bloque 12
Procesando bloque 13
Procesando bloque 14
Procesando bloque 15
Procesando bloque 16
Procesando bloque 17
Procesando bloque 18
Procesando bloque 19
Procesando bloque 20
Procesando bloque 21
Procesando bloque 22
Procesando bloque 23
Procesando bloque 24
Procesando bloque 25
Procesando bloque 26
Procesando bloque 27
Procesando bloque 28
Procesando bloque 29
Procesando bloque 30
Procesando bloque 31
Procesando bloque 32
Procesando bloque 33
Procesando bloque 34
Procesando bloque 35
Procesando bloque 36
Procesando bloque 37
Procesando bloque 38
Procesando bloque 39
Procesando bloque 40
Procesando bloque 41
Procesando bloque 42
Procesando bloque 43
Procesando bloque 44
Procesando bloque 45
Procesando bloque 46
Procesando bloque 47
Procesando bloque 48
P

In [44]:


archivo = "avisos_aemet_2021_2026_reducido_tableau.csv"

total_filas = 0
columnas = None

for chunk in pd.read_csv(
    archivo,
    chunksize=50000,
    encoding="utf-8-sig",
    low_memory=False
):
    total_filas += len(chunk)
    
    if columnas is None:
        columnas = chunk.columns.tolist()

print("Filas:", total_filas)
print("Columnas:", len(columnas))
print("Nombres de columnas:")
print(columnas)

Filas: 6738600
Columnas: 17
Nombres de columnas:
['identifier', 'sent', 'onset', 'expires', 'año', 'mes', 'mes_num', 'estacion', 'event', 'severity', 'riesgo_numerico', 'duracion_horas', 'area_desc', 'polygon', 'geocode', 'headline', 'description']
